Link bộ dữ liệu: https://huggingface.co/datasets/5CD-AI/Vietnamese-beyond-rlhf-reward-single-round-gg-translated/viewer/default/test?views%5B%5D=test

Dữ liệu Vietnamese-beyond-rlhf-reward-single-round-gg-translated do nhóm 5CD-AI phát triển và được công bố trên Hugging Face.

Bộ dữ liệu này được dịch từ các cặp hội thoại tiếng Anh sang tiếng Việt, với mục đích tạo ra các phản hồi có chất lượng cao, phù hợp với mô hình học tăng cường phản hồi.

Cấu trúc:
* train: 20.000 mẫu
* test: 5.010 mẫu

Các trường dữ liệu:
* prompt_vi: Câu hỏi hoặc yêu cầu bằng tiếng Việt.
* chosen_vi: Phản hồi được chọn bằng tiếng Việt.
* rejected_vi: Phản hồi bị loại bỏ bằng tiếng Việt.
* prompt_en: Câu hỏi hoặc yêu cầu tương ứng bằng tiếng Anh.
* chosen_en: Phản hồi được chọn bằng tiếng Anh.
* rejected_en: Phản hồi bị loại bỏ bằng tiếng Anh.

Trong bài này, các cột được sử dụng:
* prompt_en: lệnh instruction
* chosen_en: phản hồi ưa thích
* rejeccted_en: phản hồi không được ưa thích

Dữ liệu dưới đây chỉ dùng 5000 mẫu đầu tiên trong tập "train" làm dữ liệu train, và 2500 mẫu đầu tiên trong tập "test" làm dữ liệu validate

In [2]:
!pip install datasets
!pip install trl
!pip install transformers
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [3]:
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

In [ ]:
from transformers import GPT2ForSequenceClassification
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load tokenizer và model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Gán pad_token

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.config.pad_token_id = tokenizer.pad_token_id  # Gán vào config

# Nếu bạn dùng ref_model riêng, cũng làm tương tự:
re_model = GPT2LMHeadModel.from_pretrained("gpt2")
re_model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Thử train với Model khác
# model_name= "Qwen/Qwen2.5-0.5B-Instruct"
# model = AutoModelForCausalLM.from_pretrained(model_name)

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

# re_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# re_model = AutoModelForCausalLM.from_pretrained(re_model_name)
# re_model.eval()

In [18]:
# Xem trước bộ dữ liệu
# Load the dataset
dataset = load_dataset("5CD-AI/Vietnamese-beyond-rlhf-reward-single-round-gg-translated")

# Access the training and test splits
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# Use only the first 5000 samples for training and 2500 for testing
train_dataset = train_dataset
test_dataset = test_dataset

# Print information about the datasets
print("\nTraining dataset info:")
print(train_dataset)

print("\n\nTest dataset info:")
print(test_dataset)

# Get a sample from the training set
sample_train = train_dataset[0]
print("\n\nSample from training set:")
print(sample_train)

# Get a sample from the test set
sample_test = test_dataset[0]
print("\n\nSample from test set:")
print(sample_test)

# Check column names
print("\n\nColumn names in the dataset:")
print(train_dataset.column_names)

# Check the data types of the columns
print("\n\nData types of columns in the dataset:")
print(train_dataset.features)



Training dataset info:
Dataset({
    features: ['chosen_vi', 'rejected_en', 'prompt_en', 'rejected_vi', 'prompt_vi', 'chosen_en'],
    num_rows: 20000
})


Test dataset info:
Dataset({
    features: ['chosen_vi', 'rejected_en', 'prompt_en', 'rejected_vi', 'prompt_vi', 'chosen_en'],
    num_rows: 5014
})


Sample from training set:
{'chosen_vi': 'Hừm, tôi cho rằng top 10 là: Chinatown, Heat, The Untouchables, Bad Boys, True Detective, Sling Blade, Stakeout, Beverly Hills Cop, Lock, Stock, và Two Smoking Barrels, và Lethal Weapon.', 'rejected_en': 'Well, for action and comedy you have Lethal Weapon, and for action and action there’s G.I. Joe: Retaliation.', 'prompt_en': 'What are some of the best buddy cop movies?', 'rejected_vi': 'Chà, đối với hành động và hài kịch, bạn có Vũ khí sát thương, còn đối với hành động và hành động thì có GI Joe: Retaliation.', 'prompt_vi': 'Một số bộ phim cảnh sát bạn thân hay nhất là gì?', 'chosen_en': 'Hm, I’d say the top ten are:  Chinatown, Heat, The Un

In [19]:
# Trong bài này chỉ train mô hình với các prompt bằng tiếng Anh
# - prompt_en
# - chosen_en
# - rejected_en
from datasets import load_dataset

# Bước 1: Load tập train và test
dataset_train = load_dataset(
    "5CD-AI/Vietnamese-beyond-rlhf-reward-single-round-gg-translated", split="train[:5000]"
)
dataset_test = load_dataset(
    "5CD-AI/Vietnamese-beyond-rlhf-reward-single-round-gg-translated", split="test[:2500]"
)

# Bước 2: Tiền xử lý dữ liệu
def preprocess(example):
    prompt = str(example.get("prompt_en", "")).strip()
    chosen = str(example.get("chosen_en", "")).strip()
    rejected = str(example.get("rejected_en", "")).strip()

    if prompt and chosen and rejected:
        return {
            "prompt": prompt + "\n",
            "chosen": chosen + "\n",
            "rejected": rejected + "\n"
        }
    else:
        return {
            "prompt": None,
            "chosen": None,
            "rejected": None
        }

# Áp dụng tiền xử lý
dataset_train = dataset_train.map(preprocess)
dataset_train = dataset_train.filter(lambda x: x["prompt"] is not None)

dataset_test = dataset_test.map(preprocess)
dataset_test = dataset_test.filter(lambda x: x["prompt"] is not None)

# Bước 3: Giữ lại các cột cần thiết
dataset_train = dataset_train.remove_columns(
    [col for col in dataset_train.column_names if col not in {"prompt", "chosen", "rejected"}]
)
dataset_test = dataset_test.remove_columns(
    [col for col in dataset_test.column_names if col not in {"prompt", "chosen", "rejected"}]
)

# Bước 4: Kiểm tra
print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))
print(dataset_train[0])


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Train size: 4997
Test size: 2499
{'prompt': 'What are some of the best buddy cop movies?\n', 'chosen': 'Hm, I’d say the top ten are:  Chinatown, Heat, The Untouchables, Bad Boys, True Detective, Sling Blade, Stakeout, Beverly Hills Cop, Lock, Stock, and Two Smoking Barrels, and Lethal Weapon.\n', 'rejected': 'Well, for action and comedy you have Lethal Weapon, and for action and action there’s G.I. Joe: Retaliation.\n'}


In [ ]:
# define traininng args
# ft_model_name = "Qwen/Qwen2.5-0.5B-DPO"
ft_model_name = "GPT-2"  # Tên mô hình sẽ được fine-tune (huấn luyện lại)

training_args = DPOConfig(
    output_dir = ft_model_name,
    logging_steps=25,  # Ghi log sau mỗi 25 bước huấn luyện
    per_device_train_batch_size=1,  # Kích thước batch huấn luyện cho mỗi thiết bị là 1
    per_device_eval_batch_size=1,  # Kích thước batch đánh giá cho mỗi thiết bị là 1
    num_train_epochs=6,
    load_best_model_at_end=True,  # Tải mô hình tốt nhất (theo eval_loss) sau khi huấn luyện xong
    metric_for_best_model="eval_loss",  # Sử dụng chỉ số eval_loss để xác định mô hình tốt nhất
    save_strategy="epoch",  # Lưu mô hình sau mỗi epoch
    eval_strategy = "epoch",  # Đánh giá mô hình sau mỗi epoch
    eval_steps=1,  # Số bước đánh giá
)


In [ ]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
#Train mô hình
from trl import DPOTrainer

trainer = DPOTrainer(
    model=model,
    ref_model=re_model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
)
import torch
torch.cuda.empty_cache()
trainer.train()


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.780600,0.915612,0.200803,-1.034078,0.654262,1.234881,-248.699432,-218.836639,-116.338371,-118.884102
2,0.700300,1.082452,-1.073748,-2.758903,0.651861,1.685155,-261.444946,-236.084885,-115.177261,-117.875954
3,0.893500,1.126776,-1.780632,-3.624785,0.660664,1.844153,-268.513794,-244.743698,-115.352631,-117.954208
4,1.165300,1.136956,-2.373996,-4.245920,0.657463,1.871924,-274.447418,-250.955032,-114.557220,-116.436775
5,0.395400,1.171979,-2.924432,-4.840289,0.651060,1.915857,-279.951782,-256.898773,-113.893059,-115.321541
6,0.912400,1.186919,-3.206933,-5.142164,0.650660,1.935231,-282.776825,-259.917480,-113.511429,-114.721680


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=29982, training_loss=0.7677123849881864, metrics={'train_runtime': 5281.7821, 'train_samples_per_second': 5.676, 'train_steps_per_second': 5.676, 'total_flos': 0.0, 'train_loss': 0.7677123849881864, 'epoch': 6.0})

In [ ]:
# Đánh giá mô hình trên tập validation/test
metrics = trainer.evaluate()

# In các chỉ số đánh giá
print("Evaluation Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")

Evaluation Metrics:
eval_loss: 0.9156116843223572
eval_runtime: 141.5472
eval_samples_per_second: 17.655
eval_steps_per_second: 17.655
eval_rewards/chosen: 0.2008030265569687
eval_rewards/rejected: -1.0340776443481445
eval_rewards/accuracies: 0.6542617082595825
eval_rewards/margins: 1.2348806858062744
eval_logps/chosen: -248.69943237304688
eval_logps/rejected: -218.83663940429688
eval_logits/chosen: -116.33837127685547
eval_logits/rejected: -118.88410186767578
epoch: 6.0


Sau khi huấn luyện mô hình GPT-2 với phương pháp DPO đến epoch thứ 6, Thu được các chỉ số đánh giá cho thấy mô hình đã đạt được hiệu suất gần như tương đối khá tốt:
* eval_loss đạt 0.9156, cho thấy mức độ sai số trung bình trên tập kiểm tra ở mức chấp nhận được.
* Tốc độ xử lý của mô hình đạt 17.66 mẫu/giây, với thời gian đánh giá tổng cộng là 141.55 giây.
* Về phần thưởng, mô hình có reward trung bình cho mẫu được chọn là 0.2008, trong khi reward của mẫu bị từ chối là -1.0341 --> phản ánh rõ ràng rằng mô hình đã học cách phân biệt giữa các lựa chọn tốt và không tốt.
* Accuracy đạt 65.43%, chứng tỏ mô hình chọn lựa mẫu đúng theo mong đợi trong hơn một nửa số trường hợp.
* Biên độ phần thưởng (margins) giữa các lựa chọn đạt 1.2349 --> cho thấy sự khác biệt đáng kể giữa các lựa chọn đúng và sai.
* Các chỉ số log-probabilities và logits cũng cho thấy sự phân tách rõ ràng giữa lựa chọn được chấp nhận và bị từ chối, với log-probability của lựa chọn đúng thấp hơn đáng kể so với lựa chọn sai, phản ánh độ tự tin của mô hình trong việc đánh giá lựa chọn.

Tổng thể, các chỉ số này cho thấy mô hình đang học hiệu quả và có khả năng đưa ra những lựa chọn chất lượng hơn theo thời gian huấn luyện.

Đề xuất cải tiến
* Tăng số epoch huấn luyện (từ 5 đến 10).
* Dùng tập dữ liệu lớn hơn hoặc đa dạng hơn.
* Nâng cấp sang mô hình lớn hơn: Qwen, LLaMA, hoặc Mistral nếu tài nguyên cho phép.

In [ ]:
# Lưu mô hình

trainer.save_model("GPT2_DPO_6e")
tokenizer.save_pretrained("GPT2_DPO_6e")

('GPT2_DPO_6e/tokenizer_config.json',
 'GPT2_DPO_6e/special_tokens_map.json',
 'GPT2_DPO_6e/vocab.json',
 'GPT2_DPO_6e/merges.txt',
 'GPT2_DPO_6e/added_tokens.json')

In [ ]:
# Tải mô hình về máy
import shutil
shutil.make_archive("GPT2_DPO_6e", 'zip', "GPT2_DPO_6e")

from google.colab import files
files.download(f"GPT2_DPO_6e.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Load mô hình lên - Hiện tại load lên thư mục GPT2_DPO
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_path = "/content/GPT2_DPO"  # Đường dẫn đến thư mục trong Colab
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
loaded_model = GPT2LMHeadModel.from_pretrained(model_path)
loaded_model.config.pad_token_id = tokenizer.pad_token_id

In [24]:
# prompt: kiểm thử mô hình với 5 mẫu đầu tiên trong tập test

# Test mô hình với 5 mẫu đầu tiên trong tập test
for i in range(5):
    prompt = dataset_test[i]["prompt"]
    print(f"\nPrompt: {prompt}")

    # Sử dụng pipeline để tạo văn bản
    # pipeline có thể cần thêm các tham số như max_new_tokens, num_return_sequences
    # Để tạo ra nhiều phản hồi khác nhau
    pipe = pipeline("text-generation", model=loaded_model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1) # Sử dụng GPU nếu có

    # Tạo phản hồi từ mô hình
    # Bạn có thể điều chỉnh max_length hoặc max_new_tokens tùy thuộc vào yêu cầu
    generated_text = pipe(prompt, max_new_tokens=100, num_return_sequences=1)[0]["generated_text"]

    # In phản hồi được tạo ra. Cần loại bỏ phần prompt ban đầu.
    response = generated_text[len(prompt):].strip()
    print(f"\nGenerated Response:\n {response}")

    # In phản hồi được chọn và bị loại bỏ từ tập dữ liệu gốc để so sánh
    chosen_response = dataset_test[i]["chosen"]
    rejected_response = dataset_test[i]["rejected"]
    print(f"\nChosen Response (from dataset):\n {chosen_response}")
    print(f"\nRejected Response (from dataset):\n {rejected_response}")
    print("-" * 50)

Device set to use cuda:0



Prompt: Do you know why turkeys became the official food of thanksgiving?



Device set to use cuda:0



Generated Response:
 It is because humans have been eating, eating and sleeping in turkey carcasses for centuries. To help keep people from overeating, human health professionals have been able to reintroduce turkey into turkey eating to reduce fat, carbohydrates and cardiovascular risk. This has produced a delicious, versatile and affordable turkey meal, which is very nutritious.

Turkeys should be eaten with regular or in a small amount of whole turkey, like mashed potatoes and turkey and casseroles. You can also serve mashed potatoes with

Chosen Response (from dataset):
 To be honest, I don’t know anything about that. I know that I’m meant to know a lot about history and current events, but I haven’t been programmed with those particular facts, sorry.


Rejected Response (from dataset):
 I know that the American Indians were the first to celebrate the first thanksgiving, and the first European settlers came to New England in the early 1600s, and it is likely that they celebrated t

Device set to use cuda:0



Generated Response:
 A food bank is a program that provides food to low-income and poor households. We provide resources, such as diapers for children, hygiene supplies for children and meals for people with limited resources, such as cash money that can be used to keep food healthy and to provide needed housing. For more information, see Our Food Buses. We also can help people who need help finding food. Through food bank programs, all households can buy groceries, vegetables, meat for their meals and money to help them

Chosen Response (from dataset):
 Food banks offer a helping hand to people who are struggling.  They can help provide a healthy meal for a family, as well as offer support to those in need.  Many food banks also operate pantries and assistance programs, to help people improve their health and wellbeing, or support local community food and nutrition programs.


Rejected Response (from dataset):
 Food banks help people in need get access to healthy food.  This might he

Device set to use cuda:0



Generated Response:
 Emmett Shipton, "Poetry is the process of being written. It involves making connections between words, which means the meaning of words. It also involves the use of writing material to shape the way we talk, think, and think about the things in our life. Poetry involves writing in an almost impersonal way. It involves a form of writing, a process by which words can be turned in an impersonal way. The process itself is often very direct, often involving writing

Chosen Response (from dataset):
 Sure, I’ll try. Can you explain what you already know about this topic, and what you’re hoping to learn?


Rejected Response (from dataset):
 Sure, it’s certainly a broad topic. There are two common types of poems:

1) Narrative poems (which are about stories)
2) Rhyming poems (which are about things, usually fairly straightforward and specific)

Rhyming poems can be composed in different ways, too:

1) Rap - where rhyme is used for emphasis
2) Haiku - where the poem is abou

Device set to use cuda:0



Generated Response:
 A trampoline can be very effective when people are performing any of a number of tasks such as taking your breakfast on the trampoline, pulling your shoes, holding your elbows for support as you run, stepping over a sign or putting your hands on your hips or chest and then walking on it.
When it comes to exercise, trampolines can be as effective as walking, running, or sitting. The best trampoline exercises are sitting (belly, knees, trunk)

Chosen Response (from dataset):
 A good way to lose weight is to burn more calories than you consume, and the most effective way to do that is through cardio exercise.  Of course, there are many different kinds of cardio, and the right exercise for you depends on your body and preferences.  

A popular cardio exercise is jumping on a trampoline.  There are a few reasons jumping on a trampoline might be particularly effective for you:

-  The trampoline lowers the impact force of each landing, so you can jump for longer without

In [25]:
# Load mô hình cũ
base_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = GPT2LMHeadModel.from_pretrained("gpt2")
base_model.config.pad_token_id = base_tokenizer.pad_token_id

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [31]:
from transformers import pipeline

# Tạo pipeline cho cả hai mô hình
custom_generator = pipeline("text-generation", model=loaded_model, tokenizer=tokenizer)
base_generator = pipeline("text-generation", model=base_model, tokenizer=base_tokenizer)

# Danh sách các prompt để kiểm thử
prompts = [
    "Write a short story about a lost cat.",
    "Explain the theory of relativity in simple terms.",
    "What are the benefits of regular exercise?",
    "Describe a futuristic city in the year 3000.",
    "Tell me a joke about computers."
]

# Lặp qua từng prompt để tạo và so sánh đầu ra
for i, prompt in enumerate(prompts, 1):
    print(f"\n=== Prompt {i}: {prompt} ===")

    # Sinh văn bản từ mô hình đã fine-tuned
    custom_output = custom_generator(prompt, max_length=50, num_return_sequences=1, do_sample=True, truncation=True)
    print("\n* Custom Model Output:\n", custom_output[0]['generated_text'])

    print("-" * 100)

    # Sinh văn bản từ mô hình gốc
    base_output = base_generator(prompt, max_length=50, num_return_sequences=1, do_sample=True, truncation=True)
    print("* Base GPT-2 Output:\n", base_output[0]['generated_text'])
    print("=" * 100)


Device set to use cuda:0
Device set to use cuda:0



=== Prompt 1: Write a short story about a lost cat. ===

* Custom Model Output:
 Write a short story about a lost cat. Buy a cat. Buy a home. Go hunt.

The most common dog name here is "dog." Most owners think of an "American" dog at first glance. Although "America's Dog
----------------------------------------------------------------------------------------------------
* Base GPT-2 Output:
 Write a short story about a lost cat. Write a song about your cat. Write a song about your cat. It was not my name. But my cat was a young cat that left me. It only took one thing: an invitation.


=== Prompt 2: Explain the theory of relativity in simple terms. ===

* Custom Model Output:
 Explain the theory of relativity in simple terms.

The term "equation of mass" is often used by physicists to describe how much a force is between two pieces of matter. One end of the law has a constant constant value, while
----------------------------------------------------------------------------------------